In [1]:
!pip install datasets==2.19.1
!pip install seqeval

In [2]:
import datasets
import torch

# Task 1: Tải và tiền xử lý dữ liệu

In [3]:
# Tải dữ liệu CoNLL-2003
dataset = datasets.load_dataset("conll2003")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:1486: FutureWarning: The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

## Lấy dữ liệu chia thành train, valid, test

In [5]:
# 1. Trích xuất câu và nhãn số
train_sentences = dataset["train"]["tokens"]
train_tags_ids = dataset["train"]["ner_tags"]

# 2. Lấy ánh xạ số → nhãn (string)
label_names = dataset["train"].features["ner_tags"].feature.names
# Ví dụ: ['O', 'B-PER', 'I-PER', ...]

In [6]:
valid_sentences = dataset["validation"]["tokens"]
valid_tags_ids = dataset["validation"]["ner_tags"]

test_sentences = dataset['test']['tokens']
test_tags_ids = dataset['test']['ner_tags']

In [7]:
# 3. Chuyển đổi toàn bộ nhãn số sang nhãn string
train_tags = [[label_names[tag_id] for tag_id in sentence] for sentence in train_tags_ids]
valid_tags = [[label_names[tag_id] for tag_id in sentence] for sentence in valid_tags_ids]
test_tags = [[label_names[tag_id] for tag_id in sentence] for sentence in test_tags_ids]
# Kiểm tra kết quả
print("Câu đầu tiên:", train_sentences[0])
print("Nhãn số:", train_tags_ids[0])
print("Nhãn string:", train_tags[0])

Câu đầu tiên: ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
Nhãn số: [3, 0, 7, 0, 0, 0, 7, 0, 0]
Nhãn string: ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']


## Xây dựng từ điển

In [8]:
def build_vocab(sentences):
    word_to_ix = {"<PAD>": 0, "<UNK>": 1}

    for sentence in sentences:
        for word in sentence:
            # Thêm từ vào word_to_ix
            if word not in word_to_ix:
                word_to_ix[word] = len(word_to_ix)
    label_names = dataset["train"].features["ner_tags"].feature.names

    tag_to_ix = {tag: i for i, tag in enumerate(label_names)}
    tag_to_ix['<PAD>'] = len(tag_to_ix)
    return word_to_ix, tag_to_ix


In [9]:
word_to_ix, tag_to_ix = build_vocab(train_sentences)

print("Kích thước word_to_ix:", len(word_to_ix))
print("Kích thước tag_to_ix:", len(tag_to_ix))

Kích thước word_to_ix: 23625
Kích thước tag_to_ix: 10


# Task 2: Tạo PyTorch Dataset và DataLoader

In [10]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

In [11]:
vocab_size = len(word_to_ix)
num_tags = len(tag_to_ix)
PAD_WORD_ID = 0
PAD_TAG_ID = tag_to_ix["<PAD>"]
DEVICES = 'cuda'

## Dataset

In [12]:
class NERDataset(Dataset):
  def __init__(self, sentences, tags, word_to_ix, tag_to_ix):
    self.sentences = sentences
    self.tags = tags
    self.word_to_ix = word_to_ix
    self.tag_to_ix = tag_to_ix

  def __len__(self):
    return len(self.sentences)

  def __getitem__(self, idx):
    sentence = self.sentences[idx]
    tags = self.tags[idx]

    # Chuyển từ và tag sang index
    word_indices = torch.tensor(
        [self.word_to_ix.get(word, self.word_to_ix["<UNK>"]) for word in sentence],
        dtype=torch.long
    )

    tag_indices = torch.tensor(
        [self.tag_to_ix[tag] for tag in tags],
        dtype=torch.long
    )

    return word_indices, tag_indices

## DataLoader

In [13]:
def collate_fn(batch):
    word_seqs = [item[0] for item in batch]
    tag_seqs = [item[1] for item in batch]

    word_padded = pad_sequence(word_seqs, batch_first=True, padding_value=PAD_WORD_ID)
    tag_padded = pad_sequence(tag_seqs, batch_first=True, padding_value=PAD_TAG_ID)

    return word_padded, tag_padded

In [14]:
train_dataset = NERDataset(train_sentences, train_tags, word_to_ix, tag_to_ix)
valid_dataset = NERDataset(valid_sentences, valid_tags, word_to_ix, tag_to_ix)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)


# Task 3: Xây dựng mô hình RNNN

In [15]:
from torch import nn

In [16]:
class LSTMForNER(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_tags):
        super().__init__()

        # Lớp Embedding:
        # Chuyển từ ID → vector embedding (batch, seq_len, embedding_dim)
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # Lớp LSTM:
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            dropout=0.5,
            bidirectional=True,
            batch_first=True
        )

        # Lớp Linear:
        # Hidden state (hidden_dim) → số lượng nhãn UPOS
        self.linear = nn.Linear(hidden_dim * 2, num_tags)


    def forward(self, sentences):

        # Embedding: (batch, seq_len) → (batch, seq_len, embedding_dim)
        embeds = self.embedding(sentences)

        # lstm xử lý từng token
        # rnn_out: (batch, seq_len, hidden_dim)
        lstm_out, _ = self.lstm(embeds)

        # Linear dự đoán nhãn cho từng token
        # tag_scores: (batch, seq_len, num_tags)
        tag_scores = self.linear(lstm_out)

        return tag_scores


# Task 4: Huấn luyện mô hình

## Khởi tạo mô hình

In [17]:
model = LSTMForNER(
    vocab_size=vocab_size,
    embedding_dim=128,
    hidden_dim=256,
    num_tags=num_tags
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

criterion = nn.CrossEntropyLoss(ignore_index = PAD_TAG_ID)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  warnings.warn(


## Vòng lặp huấn luyện

In [18]:
def train_model(model, train_loader, valid_loader, optimizer, criterion, num_epochs=5):
    model.to(DEVICES)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for sentences, tags in train_loader:
            sentences = sentences.to(DEVICES)
            tags = tags.to(DEVICES)
            # (1) Xóa gradient cũ
            optimizer.zero_grad()

            # (2) Forward pass
            tag_scores = model(sentences)

            # (3) Tính loss
            loss = criterion(
                tag_scores.view(-1, tag_scores.size(-1)),
                tags.view(-1)
            )

            # (4) Lan truyền ngược
            loss.backward()
            # (5) Cập nhật trọng số
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_loss:.4f}")


def evaluate_loss(model, data_loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for sentences, tags in data_loader:
            sentences = sentences.to(DEVICES)
            tags = tags.to(DEVICES)

            tag_scores = model(sentences)

            loss = criterion(
                tag_scores.view(-1, tag_scores.size(-1)),
                tags.view(-1)
            )
            total_loss += loss.item()

    return total_loss / len(data_loader)

## Task 5: Đánh giá mô hình

In [19]:
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

## Hàm đánh giá và huấn luyện

In [20]:
def evaluate(model, data_loader, id2tag, tag_pad_id):
    model.eval()

    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for sentences, tags in data_loader:
            sentences = sentences.to(DEVICES)
            tags = tags.to(DEVICES)

            outputs = model(sentences)
            pred_ids = torch.argmax(outputs, dim=-1)

            for pred_seq, gold_seq in zip(pred_ids, tags):
                # Đảm bảo cả hai có cùng độ dài (lấy min length)
                min_len = min(pred_seq.size(0), gold_seq.size(0))
                pred_seq = pred_seq[:min_len]
                gold_seq = gold_seq[:min_len]

                # Tạo mask để loại bỏ padding
                mask = (gold_seq != tag_pad_id)

                # Áp dụng mask cho cả pred_seq và gold_seq
                pred_masked = pred_seq[mask]
                gold_masked = gold_seq[mask]

                correct += (pred_masked == gold_masked).sum().item()
                total   += mask.sum().item()

                # Chuyển sang list
                pred_seq_list = pred_masked.tolist()
                gold_seq_list = gold_masked.tolist()

                pred_tags = [id2tag[i] for i in pred_seq_list]
                gold_tags = [id2tag[i] for i in gold_seq_list]

                all_preds.append(pred_tags)
                all_labels.append(gold_tags)

    accuracy = correct / total if total > 0 else 0.0

    precision = precision_score(all_labels, all_preds)
    recall    = recall_score(all_labels, all_preds)
    f1        = f1_score(all_labels, all_preds)

    return accuracy, precision, recall, f1


In [21]:
def train_and_evaluate(model, train_loader, valid_loader,
                       optimizer, criterion, tag_pad_id,
                       num_epochs=5):

    model.to(DEVICES)
    best_dev_acc = 0.0

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for sentences, tags in train_loader:
            sentences = sentences.to(DEVICES)
            tags = tags.to(DEVICES)

            optimizer.zero_grad()
            outputs = model(sentences)

            loss = criterion(outputs.view(-1, outputs.size(-1)),
                             tags.view(-1))

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        train_acc,_,_,_ = evaluate(model, train_loader, label_names, tag_pad_id)
        valid_acc,_,_,_ = evaluate(model, valid_loader,label_names,  tag_pad_id)

        print(f"Epoch {epoch+1} | Loss = {avg_loss:.4f} | Train = {train_acc:.4f} | Valid = {valid_acc:.4f}")

## Huấn luyện mô hình và kết quả

In [22]:
train_and_evaluate(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    optimizer=optimizer,
    criterion=criterion,
    tag_pad_id=PAD_TAG_ID,
    num_epochs=5
)

Epoch 1 | Loss = 0.4993 | Train = 0.9245 | Valid = 0.9090
Epoch 2 | Loss = 0.2093 | Train = 0.9653 | Valid = 0.9333
Epoch 3 | Loss = 0.1067 | Train = 0.9868 | Valid = 0.9438
Epoch 4 | Loss = 0.0491 | Train = 0.9955 | Valid = 0.9476
Epoch 5 | Loss = 0.0208 | Train = 0.9990 | Valid = 0.9469


In [23]:
# Đánh giá độ chính xác cuối cùng trên tập valid
acc, precision, recall , f1 = evaluate(model, valid_loader, label_names, PAD_TAG_ID)
print('Kết quả đánh giá trên tập VALID:')
print(f"Acc: {acc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

Kết quả đánh giá trên tập VALID:
Acc: 0.9469 | Precision: 0.7675 | Recall: 0.7023 | F1: 0.7335


In [24]:
test_dataset = NERDataset(test_sentences, test_tags, word_to_ix, tag_to_ix)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

acc, precision, recall , f1 = evaluate(model, test_loader, label_names, PAD_TAG_ID)

print(f"KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST")
print(f"Acc: {acc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")


KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST
Acc: 0.9267 | Precision: 0.6851 | Recall: 0.6160 | F1: 0.6487


## Dự đoán trên câu mới

In [25]:
# Tạo từ điển ngược: index → tag
ix_to_tag = {idx: tag for tag, idx in tag_to_ix.items()}

def predict_sentence(sentence, model, word_to_ix, ix_to_tag):
    model.eval()

    # Tách câu thành các từ
    words = sentence.split()

    # Chuyển từ thành index
    word_ids = []
    for word in words:
        word_ids.append(word_to_ix.get(word, word_to_ix["<UNK>"]))

    tensor = torch.tensor(word_ids, dtype=torch.long).unsqueeze(0).to(DEVICES)

    # Dự đoán
    with torch.no_grad():
        outputs = model(tensor)
        predictions = torch.argmax(outputs, dim=-1).squeeze(0)

    pred_tags = [ix_to_tag[int(p)] for p in predictions.cpu()]

    print(f"{'Từ':<20} {'Nhãn NER':<15}")
    for word, tag in zip(words, pred_tags):
        print(f"{word:<20} {tag:<15}")

    return list(zip(words, pred_tags))

In [26]:
# Test với một số câu mới
test_sentences = [
    "VNU University is located in Hanoi",
    "Barack Obama was born in Hawaii",
    "Apple Inc is based in Cupertino California",
    "The European Union was founded in 1993"
]

print('Kết quả đánh giá trên tập VALID:')
print(f"Acc: {acc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

for test_sentence in test_sentences:
    print(f"Câu: {test_sentence}")
    predict_sentence(test_sentence, model, word_to_ix, ix_to_tag)
    print()

Kết quả đánh giá trên tập VALID:
Acc: 0.9267 | Precision: 0.6851 | Recall: 0.6160 | F1: 0.6487
Câu: VNU University is located in Hanoi
Từ                   Nhãn NER       
VNU                  B-ORG          
University           I-ORG          
is                   O              
located              O              
in                   O              
Hanoi                B-LOC          

Câu: Barack Obama was born in Hawaii
Từ                   Nhãn NER       
Barack               I-ORG          
Obama                I-ORG          
was                  O              
born                 O              
in                   O              
Hawaii               B-LOC          

Câu: Apple Inc is based in Cupertino California
Từ                   Nhãn NER       
Apple                B-ORG          
Inc                  I-ORG          
is                   O              
based                O              
in                   O              
Cupertino            B-LOC          
C